# Sample Latest Artificial NodeField Generator

Loads the most recently modified saved artificial NodeField generator and runs sampling only.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

from IPython.display import HTML
import logging
import os
import warnings

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_SILENT'] = 'true'
warnings.filterwarnings('ignore', message=r".*`isinstance\(treespec, LeafSpec\)` is deprecated.*")
warnings.filterwarnings('ignore', message=r'.*pkg_resources is deprecated as an API.*')
warnings.filterwarnings('ignore', message=r'.*Initializing zero-element tensors is a no-op.*')
warnings.filterwarnings('ignore', message=r".*does not have many workers.*")
warnings.filterwarnings('ignore', message=r'.*CrossEntropyMetric was called before the ``update`` method.*')
try:
    from pytorch_lightning.utilities.warnings import PossibleUserWarning
    warnings.filterwarnings('ignore', category=PossibleUserWarning)
except Exception:
    pass
for logger_name in ('pytorch_lightning', 'lightning', 'wandb'):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

from conditional_node_field_graph_generator.notebooks import configure_notebook
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from conditional_node_field_graph_generator.extensions.demo.latest_artificial import (
    build_artificial_plotter,
    load_latest_artificial_graph_generator,
    summarize_graphs,
)


In [ ]:
N_SAMPLES = 14
N_COLS = 7
NODE_ALPHABET_SIZE = 3
NODE_ALPHABET_KIND = 'int'
COMPONENT_SPECIFIC_ALPHABETS = True

INTERPOLATE_BETWEEN_N_SAMPLES = None
USE_ILP_DECODER = True
EDGE_PROBABILITY_THRESHOLD = None
FEASIBILITY_FILTER = None
RETURN_DECODE_STAGES = False

MODEL_DIR = SAVED_GENERATOR_ROOT
MODEL_PATTERNS = ('artificial*.pkl', '*artificial*.pkl')


In [ ]:
graph_generator, generator_model_path = load_latest_artificial_graph_generator(
    MODEL_DIR,
    patterns=MODEL_PATTERNS,
)
plot_artificial_graphs = build_artificial_plotter(
    node_alphabet_size=NODE_ALPHABET_SIZE,
    node_alphabet_kind=NODE_ALPHABET_KIND,
    component_specific_alphabets=COMPONENT_SPECIFIC_ALPHABETS,
)

print(f'Loaded latest artificial generator: {generator_model_path}')
print('model_name =', getattr(graph_generator, 'model_name', None))
print('is_fitted_ =', getattr(graph_generator, 'is_fitted_', None))
print('cached training conditioning =', len(getattr(graph_generator, 'training_graph_conditioning_', []) or []))


In [ ]:
if graph_generator.is_fitted_:
    samples = graph_generator.sample(
        n_samples=N_SAMPLES,
        interpolate_between_n_samples=INTERPOLATE_BETWEEN_N_SAMPLES,
        feasibility_filter=FEASIBILITY_FILTER,
        use_ilp_decoder=USE_ILP_DECODER,
        edge_probability_threshold=EDGE_PROBABILITY_THRESHOLD,
        return_decode_stages=RETURN_DECODE_STAGES,
    )
else:
    samples = []
    print('Sampling skipped because the loaded generator is not fitted.')


In [ ]:
if RETURN_DECODE_STAGES and isinstance(samples, dict):
    for variant_key, variant_samples in samples.items():
        print(variant_key, summarize_graphs([graph for graph in variant_samples if graph is not None]))
        plot_artificial_graphs(
            variant_samples,
            n_cols=N_COLS,
            titles=[f'{variant_key} {idx}' for idx in range(len(variant_samples))],
        );
elif samples:
    print(summarize_graphs(samples))
    plot_artificial_graphs(
        samples,
        n_cols=N_COLS,
        titles=[f'sample {idx}' for idx in range(len(samples))],
    );
else:
    print('No samples to display.')
